# Integrated Syntactic–Semantic Analysis: Keyword Extraction and Sentiment

**Authors:** Steve and Spencer
**Topic 4:** Integrated Analysis (coding task)

## Goal

1. Use **dependency parsing** (syntax) to pull out keywords and phrases from social-media style posts.
2. Score the **sentiment** (semantics) of those phrases.
3. Show why combining the two beats scoring whole sentences on their own.

## Approach

| Step | Layer | Tool |
|---|---|---|
| Parse each post into a dependency tree | Syntax | spaCy `en_core_web_sm` |
| Extract aspect + opinion pairs from specific grammatical relations | Syntax | `amod`, `nsubj`+`acomp`/`attr`, `dobj`, `neg`, `advmod`, `conj` |
| Score the opinion word, then adjust for negation and intensifiers | Semantics | VADER lexicon |
| Aggregate by aspect (battery, camera, ...) | Both | pandas + matplotlib |

## 1. Setup

Run the next cell once if the libraries are not installed yet.

In [1]:
# Uncomment and run once if needed
# !pip install spacy vaderSentiment pandas matplotlib
# !python -m spacy download en_core_web_sm

In [2]:
import re
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()
pd.set_option("display.max_colwidth", 80)
print("spaCy", spacy.__version__, "| model:", nlp.meta["name"])

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'spacy'

## 2. Sample data

A small set of made-up posts about a fictional phone, written the way people post on social media. Replace this list with your own data (for example a CSV of tweets) to reuse the notebook.

The list deliberately includes tricky cases: negation, contrast ("but"), intensifiers, and one sarcastic post.

In [ ]:
posts = [
    "The camera is amazing and the screen is gorgeous.",
    "The battery is not good, it dies before lunch.",
    "I love the fast charging but the price is ridiculous.",
    "Not bad at all, the speakers are really impressive.",
    "Terrible customer service, I waited two hours for a reply.",
    "The delivery was quick and the packaging was solid.",
    "This phone has a beautiful design and a smooth interface.",
    "The software update ruined the battery and made the phone slow.",
    "Honestly the camera is not great in low light.",
    "I hate the flimsy case that came with it.",
    "Great, another phone that overheats. Just what I needed.",   # sarcasm
    "Very disappointing battery life, but the display is excellent.",
    "The support team was helpful and friendly.",
    "The price is fair and the performance is fantastic.",
]
print(len(posts), "posts loaded")

## 3. Look at a dependency parse

Before extracting anything, it helps to see the tree for one sentence. Each token has a **dependency label** (`dep_`) and a **head** that it attaches to. These relations are what we will use as extraction rules.

In [ ]:
doc = nlp("The battery is not good, it dies before lunch.")
pd.DataFrame(
    [(t.i, t.text, t.pos_, t.dep_, t.head.text) for t in doc],
    columns=["i", "token", "pos", "dep", "head"],
)

In [ ]:
from spacy import displacy
displacy.render(nlp("The battery is not good."), style="dep", jupyter=True, options={"distance": 90})

## 4. Syntactic extraction: aspects, opinions and phrases

We look for four grammatical patterns. Each one yields an **aspect** (the thing being talked about) and an **opinion** (what is said about it).

| Pattern | Dependency rule | Example |
|---|---|---|
| Adjective modifier | adjective with `amod` attached to a noun | *gorgeous screen* |
| Copular predicate | noun `nsubj` of a verb that has an `acomp` or `attr` child | *battery is good* |
| Verb + object | opinion verb with a `dobj` noun | *love the camera* |
| Coordination | `conj` words joined to an opinion word | *quick and solid* |

We also record **negation** (a `neg` child on the opinion word or on the verb it hangs from) and **intensifiers** (an `advmod` child such as *very* or *really*).

In [ ]:
NOUNS = {"NOUN", "PROPN"}
INTENSIFIERS = {"very", "really", "extremely", "so", "incredibly", "absolutely", "super"}

def is_negated(opinion_tok, governor=None):
    """Negation can sit on the opinion word itself or on the verb it attaches to."""
    if any(c.dep_ == "neg" for c in opinion_tok.children):
        return True
    if governor is not None and any(c.dep_ == "neg" for c in governor.children):
        return True
    return False

def intensity(opinion_tok):
    return 1.3 if any(c.dep_ == "advmod" and c.lower_ in INTENSIFIERS for c in opinion_tok.children) else 1.0

def aspect_text(noun_tok):
    """Use the noun's lemma, plus a leading compound noun if there is one (customer service)."""
    comps = [c.lemma_.lower() for c in noun_tok.children if c.dep_ == "compound"]
    return " ".join(comps + [noun_tok.lemma_.lower()])

def extract_pairs(doc):
    pairs, seen = [], set()

    def add(aspect_tok, opinion_tok, rule, governor=None):
        key = (aspect_tok.i, opinion_tok.i)
        if key in seen:
            return
        seen.add(key)
        pairs.append({
            "aspect": aspect_text(aspect_tok),
            "opinion": opinion_tok.lower_,
            "negated": is_negated(opinion_tok, governor),
            "boost": intensity(opinion_tok),
            "rule": rule,
        })

    for tok in doc:
        # 1) amod: "gorgeous screen"
        if tok.dep_ == "amod" and tok.head.pos_ in NOUNS:
            add(tok.head, tok, "amod")
            for conj in tok.conjuncts:
                add(tok.head, conj, "amod+conj")

        # 2) copular: "the battery is (not) good"
        if tok.dep_ == "nsubj" and tok.pos_ in NOUNS:
            gov = tok.head
            for child in gov.children:
                if child.dep_ in ("acomp", "attr"):
                    add(tok, child, "nsubj+acomp", governor=gov)
                    for conj in child.conjuncts:
                        add(tok, conj, "nsubj+acomp+conj", governor=gov)

        # 3) verb + direct object: "love the camera", "ruined the battery"
        if tok.dep_ == "dobj" and tok.pos_ in NOUNS:
            verb = tok.head
            if verb.pos_ == "VERB":
                add(tok, verb, "verb+dobj")

    return pairs

for p in extract_pairs(nlp("The battery is not good, it dies before lunch.")):
    print(p)

### Keyword and phrase extraction

Alongside the aspect/opinion pairs we pull out **noun phrases** (`noun_chunks`) and **adjective + noun phrases**, dropping determiners and pronouns. These are the keywords we count.

In [ ]:
def keyword_phrases(doc):
    phrases = []
    for chunk in doc.noun_chunks:
        toks = [t for t in chunk if t.pos_ not in ("DET", "PRON") and not t.is_stop]
        if toks:
            phrases.append(" ".join(t.lemma_.lower() for t in toks))
    return phrases

for text in posts[:4]:
    print(text)
    print("  ->", keyword_phrases(nlp(text)))

## 5. Semantic step: sentiment of each opinion

The score for an opinion word comes from the VADER lexicon. Then the syntax-derived flags adjust it:

- **Negated:** flip the sign and shrink it a little (*not good* is mildly negative, not as strong as *terrible*).
- **Intensified:** multiply by 1.3 (*really impressive*).

Result is a value in roughly [-1, 1].

In [ ]:
NEG_FACTOR = -0.75

def opinion_score(opinion, negated, boost):
    score = vader.polarity_scores(opinion)["compound"]
    score *= boost
    if negated:
        score *= NEG_FACTOR
    return max(-1.0, min(1.0, score))

def label(score, thr=0.05):
    return "positive" if score > thr else "negative" if score < -thr else "neutral"

for args in [("good", False, 1.0), ("good", True, 1.0), ("impressive", False, 1.3), ("terrible", False, 1.0)]:
    print(args, "->", round(opinion_score(*args), 3))

## 6. Run the pipeline on all posts

In [ ]:
rows = []
for idx, text in enumerate(posts):
    doc = nlp(text)
    sent_level = vader.polarity_scores(text)["compound"]      # whole-sentence baseline
    for p in extract_pairs(doc):
        s = opinion_score(p["opinion"], p["negated"], p["boost"])
        rows.append({
            "post": idx,
            "aspect": p["aspect"],
            "opinion": ("not " if p["negated"] else "") + p["opinion"],
            "rule": p["rule"],
            "phrase_score": round(s, 3),
            "phrase_label": label(s),
            "sentence_score": round(sent_level, 3),
        })

df = pd.DataFrame(rows)
df

## 7. Results

### 7.1 Sentiment per aspect

Averaging phrase scores by aspect gives the kind of feature-level view a company would want from social media.

In [ ]:
agg = (df.groupby("aspect")
         .agg(mentions=("phrase_score", "size"), mean_sentiment=("phrase_score", "mean"))
         .sort_values("mean_sentiment"))
agg["mean_sentiment"] = agg["mean_sentiment"].round(3)
agg

In [ ]:
plot_df = agg[agg["mentions"] >= 1]
colors = ["#c0392b" if v < -0.05 else "#27ae60" if v > 0.05 else "#7f8c8d" for v in plot_df["mean_sentiment"]]

fig, ax = plt.subplots(figsize=(8, 0.45 * len(plot_df) + 1.5))
ax.barh(plot_df.index, plot_df["mean_sentiment"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean phrase sentiment (-1 to +1)")
ax.set_title("Sentiment by aspect (dependency-based extraction)")
plt.tight_layout()
plt.show()

### 7.2 Most frequent keywords and phrases

In [ ]:
all_phrases = Counter()
for text in posts:
    all_phrases.update(keyword_phrases(nlp(text)))

pd.DataFrame(all_phrases.most_common(12), columns=["phrase", "count"])

### 7.3 Why syntax helps: phrase-level vs. sentence-level sentiment

A sentence can mix opinions ("great screen but awful battery"), and a whole-sentence score averages them away. Here we compare both views for the same posts.

In [ ]:
compare = (df.groupby("post")
             .agg(sentence_score=("sentence_score", "first"),
                  phrase_view=("opinion", lambda s: ", ".join(s)),
                  phrase_mean=("phrase_score", "mean"))
             .round(3))
compare.insert(0, "text", [posts[i] for i in compare.index])
compare

In [ ]:
mixed = df.groupby("post")["phrase_label"].nunique()
mixed_posts = mixed[mixed > 1].index.tolist()
print("Posts with both positive and negative phrases:", mixed_posts)
for i in mixed_posts:
    print(f"\n[{i}] {posts[i]}")
    print("  sentence-level score:", vader.polarity_scores(posts[i])["compound"])
    print(df[df.post == i][["aspect", "opinion", "phrase_score"]].to_string(index=False))

## 8. Checking negation handling

The `neg` dependency is what makes *not good* come out negative and *not bad* come out positive. Checking those cases directly:

In [ ]:
tests = ["The battery is good.", "The battery is not good.", "The case is bad.", "The case is not bad."]
for t in tests:
    out = [(p["aspect"], ("not " if p["negated"] else "") + p["opinion"],
            round(opinion_score(p["opinion"], p["negated"], p["boost"]), 2))
           for p in extract_pairs(nlp(t))]
    print(f"{t:28s} -> {out}")

## 9. Discussion

**What worked**

- Dependency rules connected opinions to the right aspect, so *fast* attached to charging while *ridiculous* attached to price, even in the same sentence.
- The `neg` relation fixed polarity for *not good* and *not bad*, which a bag-of-words approach gets wrong.
- Aggregating by aspect turned individual posts into a feature-level summary.

**Limitations we noticed**

- **Sarcasm.** "Great, another phone that overheats" is read as positive because *great* is the only opinion the rules find. Syntax alone cannot detect irony; this needs context or a trained model.
- **Rule coverage.** We handle four patterns. Passive voice, comparatives ("better than"), and implicit opinions ("it dies before lunch") are missed or only partly captured.
- **Lexicon sentiment.** VADER is a general-purpose lexicon. Domain slang or dialect can be scored wrongly.
- **Small, invented sample.** The posts are for demonstration; conclusions should come from real data.

**Possible extensions**

- Replace the lexicon with a pretrained sentiment model (for example a transformer from Hugging Face) while still using dependency parsing to decide *which phrase* to score.
- Add rules for passives, comparatives and clause-level contrast.
- Run on a real dataset and evaluate against hand-labelled aspects.

## 10. Conclusion

Syntactic analysis told us *where to look* (which words relate to which), and semantic analysis told us *how to feel about it* (the polarity of those words). Together they gave a more precise reading of social-media text than sentence-level scoring alone, with clear gaps that point to the next improvements.

*Prepared by Steve and Spencer.*